In [ ]:
"""
DJI Video SRT Trajectory Mapper
================================
Reads DJI-style .SRT files (subtitle text files) from drone videos and creates
an interactive web map showing the flight trajectory for each video.

Features:
- Parses DJI SRT format to extract GPS coordinates and altitude
- Creates one trajectory (polyline) per video
- Adds start and end markers for each trajectory
- Uses the same color scheme as the image-based web map
- Esri Hybrid basemap (satellite + labels) as DEFAULT

Author: Farid Javadnejad
Date: 2026-04-24
"""

import os
import re
from datetime import datetime
import folium
from folium import plugins

# ============================================================================
# CONFIGURATION
# ============================================================================

# Input folder containing .srt files
INPUT_FOLDER = r"C:\Users\USFJ139860\OneDrive - WSP O365\Projects\260422 - US82 UAS Photoshoot\Videos"

# Output file (saved in same folder as SRT files)
OUTPUT_HTML = os.path.join(INPUT_FOLDER, "drone_video_trajectories.html")

# Trajectory colors (same as image-based web map)
# Colors are reused cyclically if there are more videos than colors
TRAJECTORY_COLORS = [
    '#FF0000',  # Red
    '#00FF00',  # Green
    '#0000FF',  # Blue
    '#FFFF00',  # Yellow
    '#FF00FF',  # Magenta
    '#00FFFF',  # Cyan
    '#FF8800',  # Orange
    '#8800FF',  # Purple
    '#00FF88',  # Spring Green
    '#FF0088'   # Deep Pink
]

# ============================================================================
# SRT PARSING FUNCTIONS
# ============================================================================

def parse_dji_srt_line(line):
    """
    Parse a single line from DJI SRT format to extract GPS coordinates and metadata.
    
    DJI SRT FORMAT EXAMPLE:
    [latitude: 32.965834] [longitude: -103.349189] [rel_alt: 96.500 abs_alt: 1405.827]
    
    Args:
        line (str): Single line of text from SRT file
    
    Returns:
        dict: Dictionary with latitude, longitude, altitude, timestamp or None if parsing fails
    """
    result = {
        'latitude': None,
        'longitude': None,
        'abs_altitude': None,
        'rel_altitude': None,
        'timestamp': None
    }
    
    try:
        # Extract latitude using regex pattern
        lat_match = re.search(r'\[latitude:\s*([-\d.]+)\]', line)
        if lat_match:
            result['latitude'] = float(lat_match.group(1))
        
        # Extract longitude using regex pattern
        lon_match = re.search(r'\[longitude:\s*([-\d.]+)\]', line)
        if lon_match:
            result['longitude'] = float(lon_match.group(1))
        
        # Extract absolute altitude (altitude above sea level)
        abs_alt_match = re.search(r'abs_alt:\s*([-\d.]+)', line)
        if abs_alt_match:
            result['abs_altitude'] = float(abs_alt_match.group(1))
        
        # Extract relative altitude (altitude above takeoff point)
        rel_alt_match = re.search(r'rel_alt:\s*([-\d.]+)', line)
        if rel_alt_match:
            result['rel_altitude'] = float(rel_alt_match.group(1))
        
        # Extract timestamp (format: YYYY-MM-DD HH:MM:SS.mmm)
        timestamp_match = re.search(r'(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}\.\d{3})', line)
        if timestamp_match:
            result['timestamp'] = timestamp_match.group(1)
    
    except Exception as e:
        pass  # Return None values if parsing fails
    
    return result


def parse_srt_file(srt_path):
    """
    Parse an entire DJI SRT file and extract all GPS trajectory points.
    
    DJI SRT FILE STRUCTURE:
    Each subtitle entry contains frame metadata including GPS coordinates.
    This function extracts all GPS points in chronological order.
    
    Args:
        srt_path (str): Path to the .srt file
    
    Returns:
        list: List of dictionaries containing GPS points in chronological order
    """
    trajectory_points = []
    
    try:
        with open(srt_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            
            # Split into lines and process each line
            lines = content.split('\n')
            
            for line in lines:
                # Parse line for GPS data
                point = parse_dji_srt_line(line)
                
                # Only add points with valid GPS coordinates
                if point['latitude'] is not None and point['longitude'] is not None:
                    trajectory_points.append(point)
    
    except Exception as e:
        print(f"Error parsing {os.path.basename(srt_path)}: {str(e)}")
    
    return trajectory_points


def process_srt_files(folder_path):
    """
    Process all .srt files in a folder and extract trajectories.
    
    Args:
        folder_path (str): Path to folder containing .srt files
    
    Returns:
        list: List of video trajectories, where each trajectory is a dict with:
              - 'filename': name of the SRT file
              - 'points': list of GPS points
              - 'color': assigned trajectory color
    """
    video_trajectories = []
    
    print(f"Scanning folder: {folder_path}")
    
    if not os.path.exists(folder_path):
        print(f"ERROR: Folder not found: {folder_path}")
        return video_trajectories
    
    # Find all .srt files in folder (not in subfolders)
    srt_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.srt')]
    
    if not srt_files:
        print(f"WARNING: No .srt files found in {folder_path}")
        return video_trajectories
    
    print(f"Found {len(srt_files)} .srt file(s)")
    
    # Process each SRT file
    for idx, filename in enumerate(sorted(srt_files)):
        file_path = os.path.join(folder_path, filename)
        print(f"\nProcessing: {filename}")
        
        # Parse SRT file
        points = parse_srt_file(file_path)
        
        if points:
            # Assign color (cyclical reuse if needed)
            color = TRAJECTORY_COLORS[idx % len(TRAJECTORY_COLORS)]
            
            # Create trajectory object
            trajectory = {
                'filename': filename,
                'points': points,
                'color': color,
                'video_id': idx
            }
            
            video_trajectories.append(trajectory)
            
            print(f"  ✓ Extracted {len(points)} GPS points")
            print(f"  ✓ Start: Lat {points[0]['latitude']:.6f}, Lon {points[0]['longitude']:.6f}")
            print(f"  ✓ End: Lat {points[-1]['latitude']:.6f}, Lon {points[-1]['longitude']:.6f}")
            print(f"  ✓ Assigned color: {color}")
        else:
            print(f"  ✗ No valid GPS points found")
    
    print(f"\nTotal videos with trajectories: {len(video_trajectories)}")
    return video_trajectories


# ============================================================================
# MAP GENERATION
# ============================================================================

def create_trajectory_map(video_trajectories, output_file):
    """
    Create an interactive Leaflet map with video trajectories.
    
    MAP FEATURES:
    - Esri Hybrid basemap (satellite + labels) as DEFAULT
    - One colored polyline per video trajectory
    - Start marker (green) and end marker (red) for each trajectory
    - Mutually exclusive basemaps via LayerControl
    
    Args:
        video_trajectories (list): List of video trajectory objects
        output_file (str): Output HTML file path
    """
    if not video_trajectories:
        print("No trajectories to map!")
        return
    
    # Calculate map center (average of all trajectory points)
    all_lats = []
    all_lons = []
    
    for trajectory in video_trajectories:
        for point in trajectory['points']:
            all_lats.append(point['latitude'])
            all_lons.append(point['longitude'])
    
    avg_lat = sum(all_lats) / len(all_lats)
    avg_lon = sum(all_lons) / len(all_lons)
    
    # Create base map
    m = folium.Map(
        location=[avg_lat, avg_lon],
        zoom_start=16,
        tiles=None
    )
    
    # ========================================================================
    # BASEMAP CONFIGURATION - ESRI HYBRID (DEFAULT)
    # ========================================================================
    # 
    # ESRI HYBRID BASEMAP COMPOSITION:
    # The hybrid basemap is created using TWO tile layers:
    # 1. Esri World Imagery (satellite imagery base layer)
    # 2. Esri Reference/Boundaries overlay (labels, roads, place names)
    #
    # ========================================================================
    
    # Layer 1: Esri World Imagery (satellite base) - Part of Hybrid
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Esri Hybrid',
        overlay=False,
        control=True,
        show=True  # DEFAULT visible basemap
    ).add_to(m)
    
    # Layer 2: Esri Reference Labels - Part of Hybrid
    folium.TileLayer(
        tiles='https://services.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Esri Labels',
        overlay=True,
        control=False,  # Don't show in layer control (always paired with Esri Hybrid)
        show=True  # Visible by default with Esri Hybrid
    ).add_to(m)
    
    # ========================================================================
    # ADDITIONAL BASEMAP OPTIONS (Mutually Exclusive via LayerControl)
    # ========================================================================
    
    # Esri Satellite Only (no labels)
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Satellite Only',
        overlay=False,
        control=True,
        show=False
    ).add_to(m)
    
    # OpenStreetMap (street map)
    folium.TileLayer(
        tiles='OpenStreetMap',
        name='Street Map',
        overlay=False,
        control=True,
        show=False
    ).add_to(m)
    
    # ========================================================================
    # GOOGLE HYBRID - NOT SUPPORTED IN FOLIUM
    # ========================================================================
    # Google Hybrid is NOT included because:
    # - Google Maps JavaScript API requires an API key
    # - Folium does not natively support Google Maps API integration
    # - Unofficial Google tile URLs violate Google Maps Terms of Service
    #
    # RECOMMENDATION: Esri Hybrid provides equivalent functionality
    # ========================================================================
    
    # ========================================================================
    # ADD TRAJECTORIES
    # ========================================================================
    
    for trajectory in video_trajectories:
        color = trajectory['color']
        filename = trajectory['filename']
        points = trajectory['points']
        
        # Extract coordinates for polyline
        coords = [[p['latitude'], p['longitude']] for p in points]
        
        # Calculate trajectory statistics
        start_point = points[0]
        end_point = points[-1]
        
        if start_point['abs_altitude'] and end_point['abs_altitude']:
            avg_altitude = sum([p['abs_altitude'] for p in points if p['abs_altitude']]) / len([p for p in points if p['abs_altitude']])
            min_altitude = min([p['abs_altitude'] for p in points if p['abs_altitude']])
            max_altitude = max([p['abs_altitude'] for p in points if p['abs_altitude']])
        else:
            avg_altitude = min_altitude = max_altitude = 0
        
        # Create trajectory polyline
        folium.PolyLine(
            coords,
            color=color,
            weight=3,
            opacity=0.8,
            name=f'{filename} ({len(points)} points)',
            popup=folium.Popup(f"""
                <div style="width: 280px;">
                    <h4 style="margin-bottom: 10px;">{filename}</h4>
                    <table style="width:100%; font-size: 12px;">
                        <tr><td><b>Total Points:</b></td><td>{len(points)}</td></tr>
                        <tr><td><b>Avg Altitude:</b></td><td>{avg_altitude:.1f} m</td></tr>
                        <tr><td><b>Altitude Range:</b></td><td>{min_altitude:.1f} - {max_altitude:.1f} m</td></tr>
                        <tr><td><b>Start Time:</b></td><td>{start_point['timestamp'] or 'N/A'}</td></tr>
                        <tr><td><b>End Time:</b></td><td>{end_point['timestamp'] or 'N/A'}</td></tr>
                        <tr><td><b>Trajectory Color:</b></td><td><span style="background-color:{color}; padding:2px 10px; color:white;">&nbsp;&nbsp;&nbsp;</span></td></tr>
                    </table>
                </div>
            """, max_width=300)
        ).add_to(m)
        
        # Add START marker (green circle)
        folium.CircleMarker(
            location=[start_point['latitude'], start_point['longitude']],
            radius=10,
            popup=folium.Popup(f"""
                <div style="width: 250px;">
                    <h4 style="color: green;">START: {filename}</h4>
                    <table style="width:100%; font-size: 12px;">
                        <tr><td><b>Latitude:</b></td><td>{start_point['latitude']:.6f}°</td></tr>
                        <tr><td><b>Longitude:</b></td><td>{start_point['longitude']:.6f}°</td></tr>
                        <tr><td><b>Altitude:</b></td><td>{start_point['abs_altitude']:.1f} m</td></tr>
                        <tr><td><b>Timestamp:</b></td><td>{start_point['timestamp'] or 'N/A'}</td></tr>
                    </table>
                </div>
            """, max_width=270),
            tooltip=f"START: {filename}",
            color='white',
            fillColor='green',
            fillOpacity=0.9,
            weight=3
        ).add_to(m)
        
        # Add END marker (red circle)
        folium.CircleMarker(
            location=[end_point['latitude'], end_point['longitude']],
            radius=10,
            popup=folium.Popup(f"""
                <div style="width: 250px;">
                    <h4 style="color: red;">END: {filename}</h4>
                    <table style="width:100%; font-size: 12px;">
                        <tr><td><b>Latitude:</b></td><td>{end_point['latitude']:.6f}°</td></tr>
                        <tr><td><b>Longitude:</b></td><td>{end_point['longitude']:.6f}°</td></tr>
                        <tr><td><b>Altitude:</b></td><td>{end_point['abs_altitude']:.1f} m</td></tr>
                        <tr><td><b>Timestamp:</b></td><td>{end_point['timestamp'] or 'N/A'}</td></tr>
                    </table>
                </div>
            """, max_width=270),
            tooltip=f"END: {filename}",
            color='white',
            fillColor='red',
            fillOpacity=0.9,
            weight=3
        ).add_to(m)
    
    # Add layer control (basemap switcher)
    folium.LayerControl(position='topright').add_to(m)
    
    # Add fullscreen button
    plugins.Fullscreen().add_to(m)
    
    # Add measurement tool
    plugins.MeasureControl(position='topleft').add_to(m)
    
    # Save map
    m.save(output_file)
    print(f"\n✓ Interactive map saved: {output_file}")
    print(f"  - Default basemap: Esri Hybrid (satellite + labels)")
    print(f"  - Total trajectories: {len(video_trajectories)}")


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Main execution function.
    """
    print("=" * 70)
    print("DJI VIDEO SRT TRAJECTORY MAPPER")
    print("=" * 70)
    print()
    
    # Process all SRT files in folder
    video_trajectories = process_srt_files(INPUT_FOLDER)
    
    if not video_trajectories:
        print("\n⚠ No video trajectories found. Please check:")
        print("  1. Folder path is correct")
        print("  2. Folder contains .srt files")
        print("  3. SRT files are in DJI format with GPS data")
        return
    
    # Generate map
    print("\n" + "=" * 70)
    print("GENERATING MAP")
    print("=" * 70)
    
    create_trajectory_map(video_trajectories, OUTPUT_HTML)
    
    print("\n" + "=" * 70)
    print("PROCESS COMPLETE!")
    print("=" * 70)
    print(f"\nOutput created: {OUTPUT_HTML}")
    print(f"Total videos mapped: {len(video_trajectories)}")
    print(f"\nColor assignments:")
    for traj in video_trajectories:
        print(f"  • {traj['filename']}: {traj['color']}")


if __name__ == "__main__":
    main()